# 4. Leveraged buyout

**Question.** What can a sponsor pay and still earn the target IRR under this debt structure?

**Output.** Sources & uses, debt schedule, returns bridge, IRR / MOIC, entry×exit IRR surface with cross-sections.

Debt is sized on EBITDA. A higher bid raises equity, not the term loan. Fees are outside the company.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "valuation" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from the project directory.")

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataclasses import replace
import matplotlib.pyplot as plt
import pandas as pd

from valuation import boot
from valuation import plots as viz
from valuation import report as R
from valuation.case import TOL
from valuation.statements import Line, audit, combine, forecast_statements, identity_failures, max_abs_gap

# Compact equity picker — changes cases/active.yaml for EVERY notebook.
case, UNIT = boot(picker=True)


## 1. Base-case returns


In [ ]:
from valuation.lbo import entry_equity_at_target_irr, run_lbo

result = run_lbo(case)
R.show_lbo(result, case)
assert result.converged


## 2. Structure and debt path


In [ ]:
R.table(result.sources_uses, digits=1, caption="Sources and uses at close")
viz.plot_sources_uses(result.sources_uses, UNIT)
plt.show()
viz.plot_debt_stack(result.debt_schedule, UNIT)
plt.show()
R.table(result.debt_schedule, digits=1, caption="Debt schedule")


## 3. Returns bridge


In [ ]:
R.table(result.returns_bridge, digits=1, caption="Firm-equity returns bridge")
viz.plot_returns_bridge(result.returns_bridge, UNIT)
plt.show()
R.table(result.sponsor_cashflows, digits=1, caption="Sponsor cash flows")


## 4. Bid at target IRR


In [ ]:
bid = entry_equity_at_target_irr(case)
if bid is None:
    R.note("No purchase price earns the target IRR with this debt.")
else:
    priced = run_lbo(case, entry_equity=bid)
    R.kpi_row([
        ("Bid $/sh", f"${bid / case.shares:,.2f}"),
        ("IRR at bid", f"{priced.irr:.1%}"),
        ("Entered price", f"{case.price}"),
        ("Target", f"{case.assumptions.lbo.target_irr:.0%}"),
    ])
    assert abs(priced.irr - case.assumptions.lbo.target_irr) < 1e-3


## 5. Entry × exit IRR surface

Same panel layout as the DCF: shaded surface, iso-map, cross-sections through the base entry and exit multiples.


In [ ]:
entry_grid = [7.0, 8.0, 8.5, 9.0, 10.0]
exit_grid = [7.0, 8.0, 8.5, 9.0, 10.0]
table = pd.DataFrame(index=[f"{m:.1f}x" for m in entry_grid], columns=[f"{m:.1f}x" for m in exit_grid], dtype=float)
for entry in entry_grid:
    for exit_m in exit_grid:
        trial_lbo = replace(case.assumptions.lbo, entry_ev_ebitda=entry, exit_ev_ebitda=exit_m)
        trial = replace(case, assumptions=replace(case.assumptions, lbo=trial_lbo))
        table.loc[f"{entry:.1f}x", f"{exit_m:.1f}x"] = run_lbo(trial).irr
irr_table = table * 100
R.table(irr_table, digits=1, caption="Sponsor IRR (%) — rows entry, columns exit")
b_row = viz.nearest_label(irr_table.index, case.assumptions.lbo.entry_ev_ebitda, as_multiple=True)
b_col = viz.nearest_label(irr_table.columns, case.assumptions.lbo.exit_ev_ebitda, as_multiple=True)
viz.plot_sensitivity_pair(irr_table, "Sponsor IRR (%)", "IRR %", base_row=b_row, base_col=b_col)
plt.show()

xx, yy, zz = viz.lbo_irr_surface(case)
mark_irr = float("nan") if result.irr is None else result.irr * 100
viz.plot_surface_panel(
    xx, yy, zz,
    xlabel="Entry EV/EBITDA (x)",
    ylabel="Exit EV/EBITDA (x)",
    zlabel="IRR (%)",
    title=f"{case.name} — LBO IRR sensitivity surface",
    mark=(case.assumptions.lbo.entry_ev_ebitda, case.assumptions.lbo.exit_ev_ebitda, mark_irr),
)
plt.show()
